<a href="https://colab.research.google.com/github/dhiru69-tech/ML-Techniques-Lab-Project/blob/main/notebooks/day06_scaling_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 6: Train-Test Split, Scaling and Data Leakage
**Course:** BO CDA 205, Machine Learning Techniques
**Dataset:** German Credit (1000 customers, 21 columns)
**Syllabus link:** Foundation (distance-based methods such as KNN, SVM, PCA and K-Means need scaled data)

> **Goal of today:** Split the data correctly, scale only what needs scaling, and build one preprocessing pipeline that avoids data leakage.

In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Load the German Credit dataset as a table
data = fetch_openml(name="credit-g", version=1, as_frame=True)
df = data.frame
print(df.shape)   # should print (1000, 21)

# Features (X) and target (y). bad = 1, good = 0 (as decided on Day 5)
X = df.drop(columns=['class'])
y = df['class'].map({'good': 0, 'bad': 1})

# Column names by type
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
print("Numeric:", len(num_cols), "| Categorical:", len(cat_cols))

(1000, 21)
Numeric: 7 | Categorical: 13


## 2. Train-Test Split

- **Train set (80%)**: the model learns from this
- **Test set (20%)**: kept untouched until the final evaluation

| Parameter | Why |
|---|---|
| `test_size=0.2` | 20% of rows go to the test set |
| `random_state=42` | the same split every time, so results can be reproduced |
| `stratify=y` | keeps the 70/30 good/bad ratio in both parts |

> **Rule:** split **first**, then do everything else (scaling, encoding, selection) using the training set only.

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (800, 20) | Test: (200, 20)


In [3]:
# Class ratio in each part: should be about 70% good, 30% bad in all three
print("Full :", y.value_counts(normalize=True).round(3).to_dict())
print("Train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Test :", y_test.value_counts(normalize=True).round(3).to_dict())

Full : {0: 0.7, 1: 0.3}
Train: {0: 0.7, 1: 0.3}
Test : {0: 0.7, 1: 0.3}


In [4]:
# Experiment: split WITHOUT stratify and compare the test ratio
_, _, _, y_test_ns = train_test_split(X, y, test_size=0.2, random_state=1)
print("Test ratio without stratify:", y_test_ns.value_counts(normalize=True).round(3).to_dict())

Test ratio without stratify: {0: 0.705, 1: 0.295}


## 3. Why Scaling Matters

Distance-based models compare rows by distance. A column with large numbers dominates that distance.

We check the spread of each numeric column in the **training set**.

In [5]:
# Standard deviation tells how spread out each column is
print(X_train[num_cols].std().round(2))

duration                    11.82
credit_amount             2673.55
installment_commitment       1.12
residence_since              1.11
age                         11.05
existing_credits             0.57
num_dependents               0.37
dtype: float64


## 4. StandardScaler

Formula: **z = (x − mean) / std**

After scaling, each column has **mean ≈ 0 and standard deviation ≈ 1**.

| Step | Data | Method |
|---|---|---|
| learn mean and std, then apply | train | `fit_transform` |
| apply the same mean and std | test | `transform` |

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Learn mean and std from TRAIN only, and scale train
X_train_num = scaler.fit_transform(X_train[num_cols])

# Scale TEST using the numbers learned from train (no fitting here)
X_test_num = scaler.transform(X_test[num_cols])

# Check: train mean should be about 0 and std about 1
print("Train mean:", X_train_num.mean(axis=0).round(2))
print("Train std :", X_train_num.std(axis=0).round(2))

# Test mean will NOT be exactly 0, and that is correct
print("Test mean :", X_test_num.mean(axis=0).round(2))

Train mean: [ 0.  0. -0. -0.  0.  0.  0.]
Train std : [1. 1. 1. 1. 1. 1. 1.]
Test mean : [ 0.06  0.15  0.14  0.07  0.1  -0.   -0.07]


## 5. Seeing Data Leakage

**Wrong:** fit the scaler on the full dataset. The mean and std then include the test rows.
**Right:** fit on the training set only.

The numbers differ only slightly here, but the habit matters. On bigger or noisier data, leakage can inflate scores badly.

In [7]:
# Mean learned the WRONG way (all rows) vs the RIGHT way (train rows only)
comparison = pd.DataFrame({
    'wrong (all data)': X[num_cols].mean(),
    'right (train only)': X_train[num_cols].mean()
})
comparison['difference'] = comparison['wrong (all data)'] - comparison['right (train only)']
print(comparison.round(2))

                        wrong (all data)  right (train only)  difference
duration                           20.90               20.77        0.13
credit_amount                    3271.26             3189.59       81.67
installment_commitment              2.97                2.94        0.03
residence_since                     2.84                2.83        0.02
age                                35.55               35.32        0.22
existing_credits                    1.41                1.41       -0.00
num_dependents                      1.16                1.16       -0.00


## 6. StandardScaler vs RobustScaler

`credit_amount` has large outliers. Outliers pull the mean and std, so StandardScaler is affected by them.

**RobustScaler** uses the median and IQR, which outliers barely change.

In [8]:
from sklearn.preprocessing import RobustScaler

# Scale only credit_amount both ways, fitted on train
col = ['credit_amount']
std_scaled = StandardScaler().fit(X_train[col]).transform(X_train[col])
rob_scaled = RobustScaler().fit(X_train[col]).transform(X_train[col])

# Compare the min, max and median of the scaled values
summary = pd.DataFrame({
    'standard': pd.Series(std_scaled.ravel()).describe(),
    'robust':   pd.Series(rob_scaled.ravel()).describe()
}).round(2)
print(summary)

       standard  robust
count    800.00  800.00
mean       0.00    0.34
std        1.00    1.04
min       -1.09   -0.79
25%       -0.69   -0.37
50%       -0.33    0.00
75%        0.28    0.63
max        4.77    5.28


## 7. One Preprocessing Pipeline (ColumnTransformer)

Different columns need different treatment:

| Columns | Treatment |
|---|---|
| 7 numeric | StandardScaler |
| 13 categorical | OneHotEncoder (`handle_unknown='ignore'`) |

`ColumnTransformer` applies each treatment to the right columns in one go. We **fit it on train only**, then use it on both train and test.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ],
    sparse_threshold=0      # always return a normal (dense) table
)

# Return a DataFrame with proper column names instead of a plain array
preprocess.set_output(transform="pandas")

# Fit on TRAIN only, then transform both
X_train_p = preprocess.fit_transform(X_train)
X_test_p = preprocess.transform(X_test)

print("Train processed:", X_train_p.shape)
print("Test processed :", X_test_p.shape)

Train processed: (800, 61)
Test processed : (200, 61)


In [10]:
# 1. Train and test must have identical columns
print("Same columns:", list(X_train_p.columns) == list(X_test_p.columns))

# 2. No missing values and everything numeric
print("Missing:", X_train_p.isnull().sum().sum(), X_test_p.isnull().sum().sum())
print("Non-numeric columns:", X_train_p.select_dtypes(exclude='number').shape[1])

# 3. A look at the first few columns
X_train_p.iloc[:, :9].head()

Same columns: True
Missing: 0 0
Non-numeric columns: 0


,num__duration,num__credit_amount,num__installment_commitment,num__residence_since,num__age,num__existing_credits,num__num_dependents,cat__checking_status_0<=X<200,cat__checking_status_<0
828,1.289592,1.925766,0.052281,1.053413,1.057784,-0.718745,-0.436436,0.0,1.0
997,-0.742595,-0.892853,0.942164,1.053413,0.242449,-0.718745,-0.436436,0.0,0.0
148,1.289592,0.816433,0.052281,-0.747293,-0.663479,1.045047,-0.436436,0.0,1.0
735,1.289592,0.299568,0.052281,-0.747293,-0.572886,-0.718745,-0.436436,1.0,0.0
130,2.305686,1.982655,-1.727486,-0.747293,-1.025850,-0.718745,-0.436436,1.0,0.0


## 8. Baseline: the "Always Good" Model

Before any real model, we measure the simplest one: predict **good for everyone**. Every model must beat this.

> **Remember Day 2:** this model gets about 70% accuracy but catches **zero** bad customers. Recall for the bad class shows that.

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

# 'most_frequent' always predicts the majority class (good = 0)
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train_p, y_train)
pred = baseline.predict(X_test_p)

print("Baseline accuracy:", round(accuracy_score(y_test, pred), 3))
print("Recall for bad   :", round(recall_score(y_test, pred), 3))

Baseline accuracy: 0.7
Recall for bad   : 0.0
